[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/24_Change_Management.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 变更管理（Change Management）—— 实战指南

Semantica 的 `change_management` 模块为知识图谱和本体提供版本管理、审计留痕和数据完整性校验：

- **`ChangeLogEntry`** - 标准化的变更元数据（时间戳/作者经过校验）
- **`InMemoryVersionStorage` / `SQLiteVersionStorage`** - 支持命名标签的版本快照存储
- **`compute_checksum` / `verify_checksum`** - SHA-256 完整性校验

本 notebook 跑完一个完整的 保存 → 打标签 → 校验 → 抓篡改 循环。所有输出都是对照仓库 `semantica/change_management/` 源码实际执行的结果（`pip install` cell 可能装到更新版本，行为略有差异）。

### 先看主线

14 篇说过：企业本体是活的东西，要长大、要换代。活的东西就需要版本管理——这套东西之于图谱/本体，就像 git 之于代码。五节三段，一版数据的**一生**：

**记账（第 1 节）—— 谁改了什么**

`ChangeLogEntry` 记下谁、何时、改了什么。作者必须是合法 email——构造时就校验，审计链从源头保持干净。

**存档钉版（第 2–3 节）—— 快照与标签**

2. 第 2 节 存快照：数据 + 变更日志 + SHA-256 校验和一起入库
3. 第 3 节 打标签：`release` 这类稳定名钉住版本号，消费者不必跟着版本号跑

**验身退场（第 4–5 节）—— 防伪与退役**

4. 第 4 节 验完整性：重算校验和比对——数据被改一个字符都会翻 False
5. 第 5 节 退役：`delete` 移除快照，指向它的标签要自己负责更新

与 22 篇的分工：22 管事实级溯源（这条事实从哪来），本篇管版本级变更（整张图何时被谁改了）——一个是条目账，一个是版本账。

In [1]:
!pip install -q semantica

## 1) `ChangeLogEntry` 记录*谁*改了*什么*、*何时*

`author` 必须是合法 email——数据类在构造时校验（否则抛 `ValidationError`），审计留痕因此保持干净。

In [2]:
from semantica.change_management import ChangeLogEntry

entry = ChangeLogEntry(
    timestamp="2026-08-15T09:00:00Z",
    author="demo@example.com",
    description="initial version",
)
entry

ChangeLogEntry(timestamp='2026-08-15T09:00:00Z', author='demo@example.com', description='initial version', change_id=None, related_changes=[])

## 2) 保存版本化快照

快照是一个字典：必填 `label` 加你的载荷。这里把 KG 数据、变更日志、和对除 checksum 字段外全部内容算出的 SHA-256 `checksum` 一起放入。

In [3]:
from semantica.change_management import InMemoryVersionStorage, compute_checksum

storage = InMemoryVersionStorage()

snapshot = {
    "label": "v1.0.0",
    "data": {"entities": {"acme": {"type": "Company"}}},
    "change_log": {
        "timestamp": entry.timestamp,
        "author": entry.author,
        "description": entry.description,
    },
}
snapshot["checksum"] = compute_checksum({k: v for k, v in snapshot.items() if k != "checksum"})

storage.save(snapshot)
storage.exists("v1.0.0")

True

## 3) 命名标签为发布钉住版本

`save_tag` / `get_tag` 把稳定名称（如 `release`）映射到版本号——消费者与版本号变动解耦。相当于 git tag。

In [4]:
storage.save_tag("release", "v1.0.0")

storage.get_tag("release"), [s["label"] for s in storage.list_all()]

('v1.0.0', ['v1.0.0'])

## 4) 验证完整性——并抓住篡改

`verify_checksum(snapshot)` 对快照（去掉 `checksum` 字段）重算 SHA-256 并比对。数据里一个字符被改，结果就翻成 `False`。

In [5]:
from semantica.change_management import verify_checksum

stored = storage.get("v1.0.0")
print("intact:", verify_checksum(stored))

tampered = storage.get("v1.0.0")
tampered["data"]["entities"]["acme"]["note"] = "mutated after the fact"
print("tampered:", verify_checksum(tampered))

intact: True
tampered: False


## 5) 让一个版本退役

`delete(label)` 移除快照；指向它的标签要你自己负责更新。

In [6]:
storage.delete("v1.0.0")
storage.exists("v1.0.0")

False

## 小结

### 回顾主线

一版数据的一生走完：**记账**（合法 email 的审计元数据）→ **存档钉版**（带校验和的快照 + 稳定标签）→ **验身退场**（重算比对抓篡改、delete 退役）。图谱在变、本体在变，但每一次变都有账可查、有伪可验——与 22 篇的条目账合起来，变更的账本就齐了。

| 任务 | API |
|---|---|
| 记录审计元数据 | `ChangeLogEntry(timestamp, author=email, description)` |
| 持久化一个版本 | `InMemoryVersionStorage().save({"label": ..., ...})` |
| 钉住发布名 | `save_tag("release", "v1.0.0")` / `get_tag("release")` |
| 完整性检查 | `compute_checksum(snap)` / `verify_checksum(snap)` |
| 持久化后端 | `SQLiteVersionStorage(path)` —— 同一套接口 |

另见 `semantica/change_management/change_management_usage.md`，内含管理器类（`TemporalVersionManager`、`OntologyVersionManager`）。